In [ ]:
import os
work_dir = os.path.abspath('../../..')
print(work_dir)
os.chdir(work_dir)
import sympy as sp
import numpy as np
from aiphy.experiment import default_parastructure, ExpConfig, Objstructure, DoExpType, ExpStructure, Proposition
from aiphy.experiment import concept_posx, concept_posy, concept_t, concept_dist, concept_posz
from scipy.integrate import odeint
import matplotlib.pyplot as plt

In [ ]:
from aiphy.experiment import concept_posx, concept_posy, concept_t, concept_dist, concept_posz

In [ ]:
# 4-Body Celestial Simulation using Astronomical Units
pi = 3.1415926

exp_para = {
    "x0": default_parastructure(-1e-1, 1e-1),
    "y0": default_parastructure(-1e-1, 1e-1),
    "r0": default_parastructure(2.0e-2, 2.4e-2),
    "theta0": default_parastructure(0.0, pi/2),
    "omega10": default_parastructure(2*pi/3, pi),
    "v10": default_parastructure(-5e-5, 5e-5),
    "v20": default_parastructure(-5e-5, 5e-5),
    "v30": default_parastructure(-5e-5, 5e-5),
    "v40": default_parastructure(-5e-5, 5e-5),
}

obj_info = {
    "o1": Objstructure.make_particle(5e5, 6e5),
    "o2": Objstructure.make_particle(5e5, 6e5),
    "o3": Objstructure.make_particle(5e5, 6e5),
    "o4": Objstructure.make_particle(5e5, 6e5),
    "clock": Objstructure.clock()
}

In [ ]:
data_info = [
    (concept_posx, ["o1"]),
    (concept_posy, ["o1"]),
    (concept_posz, ["o1"]),
    (concept_posx, ["o2"]),
    (concept_posy, ["o2"]),
    (concept_posz, ["o2"]),
    (concept_posx, ["o3"]),
    (concept_posy, ["o3"]),
    (concept_posz, ["o3"]),
    (concept_posx, ["o4"]),
    (concept_posy, ["o4"]),
    (concept_posz, ["o4"]),
    (concept_dist, ["o1", "o2"]),
    (concept_dist, ["o2", "o1"]),
    (concept_dist, ["o1", "o3"]),
    (concept_dist, ["o3", "o1"]),
    (concept_dist, ["o3", "o2"]),
    (concept_dist, ["o2", "o3"]),
    (concept_dist, ["o1", "o4"]),
    (concept_dist, ["o4", "o1"]),
    (concept_dist, ["o2", "o4"]),
    (concept_dist, ["o4", "o2"]),
    (concept_dist, ["o3", "o4"]),
    (concept_dist, ["o4", "o3"]),
    (concept_t, ["clock"]),
]

In [ ]:
# Gravitational constant and acceleration equations for 4-body system
G = 6.6e-11  # Real gravitational constant in SI units

acs0 = [
    sp.sympify("G*(m2*(x2-x1)/r12**3 + m3*(x3-x1)/r13**3 + m4*(x4-x1)/r14**3)"),
    sp.sympify("G*(m1*(x1-x2)/r12**3 + m3*(x3-x2)/r23**3 + m4*(x4-x2)/r24**3)"),
    sp.sympify("G*(m1*(x1-x3)/r13**3 + m2*(x2-x3)/r23**3 + m4*(x4-x3)/r34**3)"),
    sp.sympify("G*(m1*(x1-x4)/r14**3 + m2*(x2-x4)/r24**3 + m3*(x3-x4)/r34**3)"),
    sp.sympify("G*(m2*(y2-y1)/r12**3 + m3*(y3-y1)/r13**3 + m4*(y4-y1)/r14**3)"),
    sp.sympify("G*(m1*(y1-y2)/r12**3 + m3*(y3-y2)/r23**3 + m4*(y4-y2)/r24**3)"),
    sp.sympify("G*(m1*(y1-y3)/r13**3 + m2*(y2-y3)/r23**3 + m4*(y4-y3)/r34**3)"),
    sp.sympify("G*(m1*(y1-y4)/r14**3 + m2*(y2-y4)/r24**3 + m3*(y3-y4)/r34**3)")
]

In [ ]:
# Experiment configuration
t_end = 2.0
t_num = 100
error = 1e-8
exp_config = ExpConfig("celestial_4", 1, exp_para, obj_info, data_info)

In [ ]:
# Initialize random parameters for the experiment
exp_config.random_settings()

In [ ]:
# Set up initial conditions for 4-body system
x0 = exp_config.para('x0')
y0 = exp_config.para('y0')
r0 = exp_config.para('r0')
theta0 = exp_config.para('theta0')
omega10 = exp_config.para('omega10')
v10 = exp_config.para('v10')
v20 = exp_config.para('v20')
v30 = exp_config.para('v30')
v40 = exp_config.para('v40')

# Initial positions (4 bodies in a configuration)
x10 = x0 + r0*np.cos(theta0)
x20 = x0 + r0*np.cos(theta0 + 2*pi/3)
x30 = x0 + r0*np.cos(theta0 + 4*pi/3)
x40 = x0 + r0*np.cos(theta0 + pi)  # Fourth body opposite to first

y10 = y0 + r0*np.sin(theta0)
y20 = y0 + r0*np.sin(theta0 + 2*pi/3)
y30 = y0 + r0*np.sin(theta0 + 4*pi/3)
y40 = y0 + r0*np.sin(theta0 + pi)  # Fourth body opposite to first

# Initial velocities with orbital motion
vx10 = v10*np.cos(theta0) - omega10*r0*np.sin(theta0)
vy10 = v10*np.sin(theta0) + omega10*r0*np.cos(theta0)
vx20 = v20*np.cos(theta0 + 2*pi/3) - omega10*r0*np.sin(theta0 + 2*pi/3)
vy20 = v20*np.sin(theta0 + 2*pi/3) + omega10*r0*np.cos(theta0 + 2*pi/3)
vx30 = v30*np.cos(theta0 + 4*pi/3) - omega10*r0*np.sin(theta0 + 4*pi/3)
vy30 = v30*np.sin(theta0 + 4*pi/3) + omega10*r0*np.cos(theta0 + 4*pi/3)
vx40 = v40*np.cos(theta0 + pi) - omega10*r0*np.sin(theta0 + pi)
vy40 = v40*np.sin(theta0 + pi) + omega10*r0*np.cos(theta0 + pi)

In [ ]:
# Define the 4-body gravitational system
def do_experiment(x10, y10, x20, y20, x30, y30, x40, y40, 
                 vx10, vy10, vx20, vy20, vx30, vy30, vx40, vy40):
    # Initial state
    x1, y1, z1 = x10, y10, 0
    x2, y2, z2 = x20, y20, 0
    x3, y3, z3 = x30, y30, 0
    x4, y4, z4 = x40, y40, 0
    vx1, vy1, vz1 = vx10, vy10, 0
    vx2, vy2, vz2 = vx20, vy20, 0
    vx3, vy3, vz3 = vx30, vy30, 0
    vx4, vy4, vz4 = vx40, vy40, 0
    
    # Get masses
    m1 = exp_config.obj('o1').m
    m2 = exp_config.obj('o2').m
    m3 = exp_config.obj('o3').m
    m4 = exp_config.obj('o4').m
    
    def system(state, t):
        x1, y1, z1, x2, y2, z2, x3, y3, z3, x4, y4, z4, \
        vx1, vy1, vz1, vx2, vy2, vz2, vx3, vy3, vz3, vx4, vy4, vz4 = state
        
        # Calculate distances
        r12 = np.sqrt((x1-x2)**2 + (y1-y2)**2 + (z1-z2)**2)
        r13 = np.sqrt((x1-x3)**2 + (y1-y3)**2 + (z1-z3)**2)
        r14 = np.sqrt((x1-x4)**2 + (y1-y4)**2 + (z1-z4)**2)
        r23 = np.sqrt((x2-x3)**2 + (y2-y3)**2 + (z2-z3)**2)
        r24 = np.sqrt((x2-x4)**2 + (y2-y4)**2 + (z2-z4)**2)
        r34 = np.sqrt((x3-x4)**2 + (y3-y4)**2 + (z3-z4)**2)
        
        # Calculate accelerations for body 1
        ax1 = G * (m2 * (x2-x1) / r12**3 + m3 * (x3-x1) / r13**3 + m4 * (x4-x1) / r14**3)
        ay1 = G * (m2 * (y2-y1) / r12**3 + m3 * (y3-y1) / r13**3 + m4 * (y4-y1) / r14**3)
        az1 = G * (m2 * (z2-z1) / r12**3 + m3 * (z3-z1) / r13**3 + m4 * (z4-z1) / r14**3)
        
        # Calculate accelerations for body 2
        ax2 = G * (m1 * (x1-x2) / r12**3 + m3 * (x3-x2) / r23**3 + m4 * (x4-x2) / r24**3)
        ay2 = G * (m1 * (y1-y2) / r12**3 + m3 * (y3-y2) / r23**3 + m4 * (y4-y2) / r24**3)
        az2 = G * (m1 * (z1-z2) / r12**3 + m3 * (z3-z2) / r23**3 + m4 * (z4-z2) / r24**3)
        
        # Calculate accelerations for body 3
        ax3 = G * (m1 * (x1-x3) / r13**3 + m2 * (x2-x3) / r23**3 + m4 * (x4-x3) / r34**3)
        ay3 = G * (m1 * (y1-y3) / r13**3 + m2 * (y2-y3) / r23**3 + m4 * (y4-y3) / r34**3)
        az3 = G * (m1 * (z1-z3) / r13**3 + m2 * (z2-z3) / r23**3 + m4 * (z4-z3) / r34**3)
        
        # Calculate accelerations for body 4
        ax4 = G * (m1 * (x1-x4) / r14**3 + m2 * (x2-x4) / r24**3 + m3 * (x3-x4) / r34**3)
        ay4 = G * (m1 * (y1-y4) / r14**3 + m2 * (y2-y4) / r24**3 + m3 * (y3-y4) / r34**3)
        az4 = G * (m1 * (z1-z4) / r14**3 + m2 * (z2-z4) / r24**3 + m3 * (z3-z4) / r34**3)
        
        return [vx1, vy1, vz1, vx2, vy2, vz2, vx3, vy3, vz3, vx4, vy4, vz4,
                ax1, ay1, az1, ax2, ay2, az2, ax3, ay3, az3, ax4, ay4, az4]
    
    # Time array and initial state
    t = np.linspace(0, t_end, t_num)
    state0 = [x1, y1, z1, x2, y2, z2, x3, y3, z3, x4, y4, z4,
              vx1, vy1, vz1, vx2, vy2, vz2, vx3, vy3, vz3, vx4, vy4, vz4]
    
    # Solve the differential equation
    solution = odeint(system, state0, t)
    
    return solution, t

In [ ]:
# Run the 4-body simulation
solution, t = do_experiment(x10, y10, x20, y20, x30, y30, x40, y40, 
                         vx10, vy10, vx20, vy20, vx30, vy30, vx40, vy40)

In [ ]:
# Visualize the 4-body trajectories
plt.figure(figsize=(12, 10))
plt.plot(solution[:, 0], solution[:, 1], label='Body 1', linewidth=2)
plt.plot(solution[:, 3], solution[:, 4], label='Body 2', linewidth=2)
plt.plot(solution[:, 6], solution[:, 7], label='Body 3', linewidth=2)
plt.plot(solution[:, 9], solution[:, 10], label='Body 4', linewidth=2)

# Mark initial positions
plt.plot(solution[0, 0], solution[0, 1], 'ro', markersize=8, label='Body 1 Start')
plt.plot(solution[0, 3], solution[0, 4], 'go', markersize=8, label='Body 2 Start')
plt.plot(solution[0, 6], solution[0, 7], 'bo', markersize=8, label='Body 3 Start')
plt.plot(solution[0, 9], solution[0, 10], 'mo', markersize=8, label='Body 4 Start')

plt.xlabel('X Position (astronomical units)', fontsize=12)
plt.ylabel('Y Position (astronomical units)', fontsize=12)
plt.title('4-Body Celestial Motion Simulation', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()